# Q3. (Optional) LangGraph, LangChain등을 활용해서 해당 유형에 알맞은 메세지를 출력하는 암보험 상담 챗봇을 만들어봅시다."

### In this notebook, I have created a simple chatbot which would provide the customer responses that are based on the query classification from Q2. For the pipeline, I have implemented ChagGPT-4o-mini for LLM, and Langchain for RAG. The main reason why I chose Langchain instead of Langgraph is because Langgraph does not provide much benefit in my current circumstances.<br>

###In my opinion, I believe Langgraph comes in handy when:

<br>

1. There are multiple tools/agents to be utilized
2. The chatbot requires multiple steps for verifying whether the query is not clear or the output response is not satisfying.
3. Keeping track of states (ex: previous chats or multiple retrieval)

<br>

### However, due to limited amount of time & resources (especially documents for RAG) to prepare multiple tools, I have decided to use Langchain to build the chatbot pipeline. <br>


### Although Langgraph has not been used, I would like to show that the chat system I made produces appropriate responses based on the classified queries and RAG. The results show that it is capable of retrieving the right document from the vector db and utilizes well for answering insurance specific inquiries.




## Dependencies

In [ ]:
!pip install -U langchain_community pypdf langchain-text-splitters \
langchain_core langchain-ollama pymupdf langchain-pymupdf4llm pymupdf-layout \
pydantic langchain-openai openai chromadb langchain_huggingface

### For the pipeline, I will be mainly using langchain to retrieve information from "한화 100세 암치료보장보험 상품요약서" (stored at my google drive to fetch the document to google colab) and use it to answer user's questions regarding the insurance plans.

[한화 100세 암치료보장보험 상품요약서 링크](https://www.hwgeneralins.com/notice/ir/product-ing01.do)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
import os

folder_path = '/content/drive/MyDrive/Hanhwa_files'

if os.path.exists(folder_path):
    if folder_path not in sys.path:
        sys.path.append(folder_path)
    print(f"Path added: {folder_path}")
else:
    print(f"Error: The path {folder_path} does not exist.")

Path added: /content/drive/MyDrive/Hanhwa_files


In [ ]:
doc_path = "/content/drive/MyDrive/Hanhwa_files/hanhwa_100cancer_product_info.pdf"

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader as pdf_loader
from langchain_pymupdf4llm import PyMuPDF4LLMLoader
from pprint import pprint

### There are various ways to load pdf files, but after several trials, I believe that PyMuPDF4LLMLoder works best for most of them (Due to some dependency conflicts, I did not set parameters related to image processing - though there are no images in the pdf document that is being used).  

In [ ]:
def load_documents():
  document_loader = PyMuPDF4LLMLoader(doc_path,
                               mode='page',
                               )
  return document_loader.load()

documents = load_documents()

In [ ]:
pprint(documents[0].page_content)

('확인필-제2025-상품개발1-기타(안내,교육)05008L-전사(25.10.21~26.10.20) \n'
 '\n'
 '## **한화 100세 암치료보장보험 무배당 상품요약서** \n'
 '\n'
 '**==> picture [117 x 52] intentionally omitted <==**\n'
 '\n'
 '**==> picture [117 x 52] intentionally omitted <==**\n'
 '\n'
 '<Q & A> \n'
 '\n'
 '## Q. 해약환급금 일부지급형에 대해 설명해주세요. \n'
 '\n'
 '- 【2종(납입면제형, 납입후50%해약환급금지급형) \n'
 '\n'
 '- ☞ 가. 납입후50%해약환급금지급형은 보험료 납입기간 중 계약이 해지될 경우 해약환급금을 지급하지 않으며, 보험료 납입이 완료되고 '
 '납입기간이 종료된 이후 계약이 해지되는 경우에는 기본형 해약환급금의 50%를 지급하는 상 품을 말합니다. 단, 2종(납입면제형, '
 '납입후50%해약환급금지급형)에 부가되는 갱신형 특별약관 및 독립특별약 관은 계약이 해지될 경우 해당 특별약관의 해약환급금을 100% '
 '지급합니다. \n'
 '\n'
 '- 나. 납입후50%해약환급금지급형의 경우 보험종목, 보험기간, 보험료 납입기간, 피보험자의 변경, 보험가입금액의 증액 및 보장의 추가는 '
 '신청할 수 없습니다. \n'
 '\n'
 '- 다. 회사는 납입후50%해약환급금지급형 계약을 체결할 때 계약자에게 납입후50%해약환급금지급형 내용에 대한 충분한 설명을 하고 별도의 '
 '확인서를 받습니다. 이 때 회사는 기본형과의 보험료 및 해약환급금(환급률 포함) 수준을 비교·안내하여 드립니다. \n'
 '\n'
 '① 보험료 납입이 완료되고 납입기간 종료 후 계약 해지 시 1종(납입면제형, 기본형) 해약환급금의 2종(납입면제형, 50% 지급 '
 '납입후50%해약환급금지급형) · ② 가입시 1종(납입면제형, 기본형) 과의 보험료 및 해약환급금(환급률 포함) 수준 비교

In [ ]:
pprint(documents[0].metadata)

{'author': 'U+ U+ U+ U+',
 'creationDate': "D:20251021141620+09'00'",
 'creationdate': '2025-10-21T14:16:20+09:00',
 'creator': 'Hwp 2020 11.0.0.8362',
 'file_path': '/content/drive/MyDrive/Hanhwa_files/hanhwa_100cancer_product_info.pdf',
 'format': 'PDF 1.4',
 'keywords': '',
 'modDate': "D:20251022082709+09'00'",
 'moddate': '2025-10-22T08:27:09+09:00',
 'page': 0,
 'producer': 'Hancom PDF 1.3.0.546',
 'source': '/content/drive/MyDrive/Hanhwa_files/hanhwa_100cancer_product_info.pdf',
 'subject': '',
 'title': '',
 'total_pages': 35,
 'trapped': ''}


### As we can see the content & metadata are well extracted.

### Now we could observe that there are some unwanted headers and footers at each page so we will be deleting them from the document.

In [ ]:
import re
from typing import List, Tuple

# Unwanted header: 확인필-제2025-상품개발1-기타(안내,교육)05008L-전사(25.10.21~26.10.20)
HEADER_RE = re.compile(
    r"""
    (?m)
    ^\s*
    확인필-제\d{4}
    -상품개발\d+
    -[^\n]*?
    전사\(
        \d{2}\.\d{2}\.\d{2}~\d{2}\.\d{2}\.\d{2}
    \)
    \s*\n+
    """,
    re.VERBOSE,
)

# Unwanted Footer: 한화 100세 암치료보장보험 무배당
FOOTER_RE = re.compile(
    r"""
    (?ms)
    \n+
    \s*
    (?:\d+\s*)?
    한화\s*100세\s*암치료보장보험\s*무배당
    (?:\s*\d+)?
    \s*
    \n*\s*$
    """,
    re.VERBOSE,
)

# Excessive blank line deletion
BLANKS_RE = re.compile(r"\n{3,}")


def strip_header_footer(text):
    had_header = bool(HEADER_RE.search(text))
    had_footer = bool(FOOTER_RE.search(text))

    text = HEADER_RE.sub("", text, count=1)
    text = FOOTER_RE.sub("", text, count=1)

    text = BLANKS_RE.sub("\n\n", text).strip() + "\n"
    return text, had_header, had_footer

missing_header_pages = []
missing_footer_pages = []

for i, doc in enumerate(documents):
    cleaned_text, had_header, had_footer = strip_header_footer(doc.page_content)

    doc.page_content = cleaned_text

    if not had_header:
        missing_header_pages.append(i)
    if not had_footer:
        missing_footer_pages.append(i)

print("Pages without header match: {}\nPages without footer match: {}".format(missing_header_pages, missing_footer_pages))



Pages without header match: []
Pages without footer match: []


In [ ]:
pprint(documents[-2].page_content)

('- - 1종(납입면제형, 기본형)의 경우 회사는 금융감독원장이 인가한 산출기준에 따라 계산한 이 보험의 계약자적립액 에서 해약공제액을 '
 '공제한 금액을 해약환급금으로 지급하여 드립니다. \n'
 '\n'
 '- - 2종(납입면제형, 납입후50%해약환급금지급형)의 경우 보험료 납입기간 중 계약이 해지될 경우 해약환급금을 지급 하지 않습니다. '
 '다만, 보험료 납입이 완료되고 납입기간이 종료된 이후 계약이 해지되는 경우 1종(납입면제형, 기 본형) 해약환급금의 50%를 '
 '지급합니다.(단, 독립특별약관, 갱신형 특별약관은 "납입후50%해약환급금지급형"이 아 니며, 독립특별약관 및 갱신형 특별약관을 가입하는 '
 '경우 보험료 납입기간 중이라도 중도해지시 해약환급금이 발 생할 수 있습니다) \n'
 '\n'
 '## 바-2. 해약환급금 \n'
 '\n'
 '- ㅇ 가입기준 : 1종(납입면제형,기본형) 남자 40세, 상해1급, 월납 50,000원, 20년납 100세만기 ㅇ 보통약관 : '
 '암(4대유사암제외)진단비 1,000만원 ㅇ 특별약관 : 상해사망 2,000만원, 4대유사암진단비 세부보장별 200만원, '
 '암(4대유사암제외)진단후특정치료비(진단 후10년,연간1회한) 100만원, 특정유사암진단후특정치료비(진단후10년,연간1회한) 100만원 \n'
 '\n'
 '(단위 : 원, %) \n'
 '\n'
 '|구 분|납입보험료|최저보증이율|최저보증이율|적용이율|적용이율|적용이율|적용이율|\n'
 '|---|---|---|---|---|---|---|---|\n'
 '|||||평균공시이율||공시이율||\n'
 '|||해약환급금|환급률|해약환급금|환급률|해약환급금|환급률|\n'
 '|1년|600,000|11,038|1.8|12,519|2.0|12,519|2.0|\n'
 '|3년|1,800,000|1,022,338|56.7|1,035,112|57.5|1,035,112|57.5|\n'
 '|5년|3,000,000|2,068,715|68.9|2,104,225

### All unwanted headers and footers are now handled.

### Now for effective RAG, I tried to extract the section titles that could be done via Regex. The titles will then be placed as a field in metadata for better categorization (Here, I have set 2 fields: block_category (ex: Q&A, 가입자격제한 및 상품별 특이사항) & block_title (ex: 해약환급금 일부지급형에 대해 설명해주세요 [under Q&A])). Additionally, the contents separated by each section will be stored inside a new document list (pdf_docs).

### **However, in my opinion for the best RAG performance the metadata filling / categorization should be mostly done by hand as there are some limits to using Regex & LLM for the task (especially when handling documents that does not have a clear notations that shows the hierarchies of the contents)

In [ ]:
import re
from copy import deepcopy
from typing import List, Optional
from langchain_core.documents import Document


DASH = r"[-‐-‒–—―－]"
DOT  = r"[\.．]"

QA_CATEGORY_RE = re.compile(r"(?m)^\s*<\s*Q\s*&?\s*A\s*>\s*$")

# "가.", "나.", ...
GEN_CATEGORY_RE = re.compile(
    rf"^\s*(?:#+\s*)?(?:\*{{0,3}}\s*)?(?P<letter>[가나다라마바사아자차카타파하])\s*{DOT}\s*(?P<body>.+?)\s*(?:\*{{0,3}})?\s*$"
)

LEADING_MD_RE = re.compile(r"^\s*(?:[-•·]\s*)?(?:#+\s*)?")
EDGE_EMPH_RE   = re.compile(r"^\*{1,3}\s*|\s*\*{1,3}$")
EDGE_BRACKETS_RE = re.compile(r"^\[\s*|\s*\]$")

def normalize_line_for_marker(line):
    s = line.strip()
    s = LEADING_MD_RE.sub("", s).strip()
    s = EDGE_EMPH_RE.sub("", s).strip()

    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1].strip()
    else:
        s = EDGE_BRACKETS_RE.sub("", s).strip()
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [ ]:
# Q title
Q_TITLE_RE = re.compile(rf"^Q\s*{DOT}\s*(?P<body>.+)$")

# Korean subsection title: 가-1., 나-1., ...
GEN_TITLE_RE = re.compile(
    rf"^(?P<letter>[가나다라마바사아자차카타파하])\s*{DASH}\s*(?P<num>\d+)\s*(?:{DOT}\s*)?(?P<body>.+)$"
)

INLINE_FIX_RE = re.compile(r"(?<!\n)(\s*[-•]?\s*Q\s*[\.．])")

def strip_space(text):
    return re.sub(r"\s+", " ", text).strip()

def detect_category(line):
    if QA_CATEGORY_RE.match(line):
        return "<Q&A>"
    m = GEN_CATEGORY_RE.match(line)
    if m:
        letter = m.group("letter")
        body = strip_space(m.group("body"))
        return f"{letter}. {body}"
    return None

def detect_title(line):
    s = normalize_line_for_marker(line)

    m = Q_TITLE_RE.match(s)
    if m:
        return f"Q. {strip_space(m.group('body'))}"

    m = GEN_TITLE_RE.match(s)
    if m:
        letter = m.group("letter")
        num = m.group("num")
        body = strip_space(m.group("body"))
        return f"{letter}-{num}. {body}"

    return None

In [ ]:
def Inline_fix(text):
    return INLINE_FIX_RE.sub(r"\n\1", text)

def split_into_blocks(documents: List[Document]) -> List[Document]:
    pdf_docs: List[Document] = []

    current_category: str = ""
    current_title: str = ""
    buffer: List[str] = []
    start_page_idx: Optional[int] = None
    last_page_idx: Optional[int] = None

    base_meta = deepcopy(documents[0].metadata) if documents else {}

    def flush(end_page_idx: int):
        nonlocal buffer, start_page_idx, last_page_idx, current_category, current_title, pdf_docs

        content = "\n".join(buffer).strip()
        buffer = []
        if not content:
            return

        meta = deepcopy(base_meta)
        meta.update({
            "block_category": current_category,
            "block_title": current_title,
            "start_page": (start_page_idx + 1) if start_page_idx is not None else None,
            "end_page": end_page_idx + 1,
        })
        pdf_docs.append(Document(page_content=content + "\n", metadata=meta))

        start_page_idx = None
        last_page_idx = None

    for page_idx, doc in enumerate(documents):
        raw = Inline_fix(doc.page_content)

        for line in raw.splitlines():
            new_cat = detect_category(line)
            if new_cat:
                if last_page_idx is None:
                    last_page_idx = page_idx
                flush(end_page_idx=last_page_idx)

                current_category = new_cat
                current_title = ""
                start_page_idx = page_idx
                last_page_idx = page_idx
                continue

            new_title = detect_title(line)
            if new_title:
                if last_page_idx is None:
                    last_page_idx = page_idx
                flush(end_page_idx=last_page_idx)

                current_title = new_title
                start_page_idx = page_idx
                last_page_idx = page_idx
                continue

            if start_page_idx is None:
                start_page_idx = page_idx
            last_page_idx = page_idx
            buffer.append(line)

    if last_page_idx is not None:
        flush(end_page_idx=last_page_idx)

    return pdf_docs

In [ ]:
pdf_docs = split_into_blocks(documents)
print("num blocks: {}\n".format(len(pdf_docs)))
pprint(pdf_docs[1].metadata)


num blocks: 20

{'author': 'U+ U+ U+ U+',
 'block_category': '<Q&A>',
 'block_title': '',
 'creationDate': "D:20251021141620+09'00'",
 'creationdate': '2025-10-21T14:16:20+09:00',
 'creator': 'Hwp 2020 11.0.0.8362',
 'end_page': 1,
 'file_path': '/content/drive/MyDrive/Hanhwa_files/hanhwa_100cancer_product_info.pdf',
 'format': 'PDF 1.4',
 'keywords': '',
 'modDate': "D:20251022082709+09'00'",
 'moddate': '2025-10-22T08:27:09+09:00',
 'page': 0,
 'producer': 'Hancom PDF 1.3.0.546',
 'source': '/content/drive/MyDrive/Hanhwa_files/hanhwa_100cancer_product_info.pdf',
 'start_page': 1,
 'subject': '',
 'title': '',
 'total_pages': 35,
 'trapped': ''}


In [ ]:
print(pdf_docs[10].page_content[:300], "\n")
pprint(pdf_docs[10].metadata)

보험계약자가 납입하는 보험료는 보험사고 발생시 보험금지급을 위한 위험보험료, 만기시 환급금을 지급하기 위한 적립보험료, 보험회사의 사업경비를 위한 부가보험료 등으로 구성됩니다. 단, 2종(납입면제형, 납입후50%해약환급 금지급형)의 경우 적립보험료는 운영하지 않습니다.
 

{'author': 'U+ U+ U+ U+',
 'block_category': '다. 보험료 산출기초 및 공시이율',
 'block_title': '다-1. 보험료의 구성',
 'creationDate': "D:20251021141620+09'00'",
 'creationdate': '2025-10-21T14:16:20+09:00',
 'creator': 'Hwp 2020 11.0.0.8362',
 'end_page': 32,
 'file_path': '/content/drive/MyDrive/Hanhwa_files/hanhwa_100cancer_product_info.pdf',
 'format': 'PDF 1.4',
 'keywords': '',
 'modDate': "D:20251022082709+09'00'",
 'moddate': '2025-10-22T08:27:09+09:00',
 'page': 0,
 'producer': 'Hancom PDF 1.3.0.546',
 'source': '/content/drive/MyDrive/Hanhwa_files/hanhwa_100cancer_product_info.pdf',
 'start_page': 32,
 'subject': '',
 'title': '',
 'total_pages': 35,
 'trapped': ''}


### We now see that metadata now contains the block_category and block_title along with the content being well separated according to the block_title.

##Embedding

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

### For Korean embedding, I was able to find a nice document which organized & ranked all the SOTA embeddings by performances. [Korean Embedding List](https://github.com/OnAnd0n/ko-embedding-leaderboard)

### For this, I chose bge-m3 embedding model as it shows decent performance - have relatively high scores on many of the evaluation metrics.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

model_kwargs = model_kwargs = {'device':'cuda', 'trust_remote_code': True}
embeddings_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs=model_kwargs,
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

### For splitting, each chunks will be divided into approximately 1500 characters with overlap of approximately 150 characters (10%).

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
tokenizer.model_max_length = 20000

text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer = tokenizer,
    chunk_size = 1500,
    chunk_overlap  = 150,
)

split_docs = text_splitter.split_documents(pdf_docs[1:])
print(len(split_docs), "\n")
print(split_docs[0].page_content, "\n")
pprint(split_docs[0].metadata)


49 

## 

{'author': 'U+ U+ U+ U+',
 'block_category': '<Q&A>',
 'block_title': '',
 'creationDate': "D:20251021141620+09'00'",
 'creationdate': '2025-10-21T14:16:20+09:00',
 'creator': 'Hwp 2020 11.0.0.8362',
 'end_page': 1,
 'file_path': '/content/drive/MyDrive/Hanhwa_files/hanhwa_100cancer_product_info.pdf',
 'format': 'PDF 1.4',
 'keywords': '',
 'modDate': "D:20251022082709+09'00'",
 'moddate': '2025-10-22T08:27:09+09:00',
 'page': 0,
 'producer': 'Hancom PDF 1.3.0.546',
 'source': '/content/drive/MyDrive/Hanhwa_files/hanhwa_100cancer_product_info.pdf',
 'start_page': 1,
 'subject': '',
 'title': '',
 'total_pages': 35,
 'trapped': ''}


In [ ]:
print(split_docs[10].page_content, "\n")
pprint(split_docs[10].metadata)

## 다) 갱신형 특별약관 

|구    분||보험기간|납입기<br>간|가입나이|
|---|---|---|---|---|
|암(4대유사암제외)진단후특정치료비(진단후10년,연간1회한)(갱신형)<br>특정유사암진단후특정치료비(진단후10년,연간1회한)(갱신형)<br>암(4대유사암제외)진단후특정치료비(상급종합병원Ⅱ)(진단후10년,연간1<br>회한)(갱신형)<br>특정유사암진단후특정치료비(상급종합병원Ⅱ)(진단후10년,연간1회한)(갱<br>신형)<br>암(4대유사암제외)진단후종합병원특정치료비(상급종합병원제외)(진단후1<br>0년,연간1회한)(갱신형)<br>특정유사암진단후종합병원특정치료비(상급종합병원제외)(진단후10년,연<br>간1회한)(갱신형)<br>하이클래스Ⅰ암(특정유사암포함)진단후특정치료비(진단후10년,연간1회한<br>)(갱신형)<br>하이클래스Ⅰ암(특정유사암포함)항암약물치료비(진단후10년,연간1회한)(<br>갱신형)<br>하이클래스Ⅱ암(특정유사암포함)특정치료비(연간1회한)(갱신형)<br>하이클래스Ⅲ암(4대유사암제외)항암약물치료비(연간1회한)(갱신형)<br>하이클래스Ⅲ특정유사암항암약물치료비(연간1회한)(갱신형)<br>암(특정유사암포함)항암중입자방사선치료비(1회한)(갱신형)<br><br>|최초계약|10/20년만기|전기납|15세 ~80세|
||갱신계약|10/20년만기||[15+보험기간]세<br>~ <br>[100-보험기간]<br>세|
|||1~9,<br>11~19년만기||[90-보험기간]세,<br>[100-보험기간]<br>세|
|구    분||보험기간|납입기<br>간|가입나이|
|---|---|---|---|---|
|암(4대유사암제외)특정치료비(종합병원)(각연간1회한)(갱신형)<br>4대유사암특정치료비(종합병원)(각연간1회한)(갱신형)|최초계약|10/20년만기|전기납|20세 ~80세|
||갱신계약|10/20년만기||[20+보험기간]세<br>~ <br>[100-보험기간]<br>세|
|||1~9,<br>11~19년만기||[90-보험기간]세,<br>[100-보험기간]<br>

In [ ]:
pprint(split_docs[2].page_content) # checking for very small chunks

('## ☞ 가. 자동갱신 특약으로 운영되는 담보는 다음과 같습니다. \n'
 '\n'
 '암(4대유사암제외)진단후특정치료비(진단후10년,연간1회한)(갱신형) 특정유사암진단후특정치료비(진단후10년,연간1회한)(갱신형) '
 '통합암(4대유사암제외)진단비Ⅲ(갱신형) 암(4대유사암제외)진단후특정치료비(상급종합병원Ⅱ)(진단후10년,연간1회한)(갱신형) '
 '특정유사암진단후특정치료비(상급종합병원Ⅱ)(진단후10년,연간1회한)(갱신형) '
 '암(4대유사암제외)진단후종합병원특정치료비(상급종합병원제외)(진단후10년,연간1회한)(갱신형) '
 '특정유사암진단후종합병원특정치료비(상급종합병원제외)(진단후10년,연간1회한)(갱신형) '
 '하이클래스Ⅰ암(특정유사암포함)진단후특정치료비(진단후10년,연간1회한)(갱신형) '
 '하이클래스Ⅰ암(특정유사암포함)항암약물치료비(진단후10년,연간1회한)(갱신형) 하이클래스Ⅱ암(특정유사암포함)특정치료비(연간1회한)(갱신형) '
 '하이클래스Ⅲ암(4대유사암제외)항암약물치료비(연간1회한)(갱신형) 하이클래스Ⅲ특정유사암항암약물치료비(연간1회한)(갱신형) '
 '하이클래스Ⅲ암(4대유사암제외)항암방사선치료비(연간1회한)(갱신형) 하이클래스Ⅲ특정유사암항암방사선치료비(연간1회한)(갱신형) '
 '암(4대유사암제외)특정치료비(종합병원)(각연간1회한)(갱신형) 4대유사암특정치료비(종합병원)(각연간1회한)(갱신형) '
 '암(4대유사암제외)특정치료비(상급종합병원Ⅲ)(각연간1회한)(갱신형) 4대유사암특정치료비(상급종합병원Ⅲ)(각연간1회한)(갱신형) '
 '암(특정유사암포함)후유장해(3-100%)(갱신형) 암(특정유사암포함)항암중입자방사선치료비(1회한)(갱신형) '
 '암(갑상선암및전립선암제외)다빈치로봇수술비(1회한)(갱신형) 갑상선암및전립선암다빈치로봇수술비(1회한)(갱신형) '
 '자궁및난소특정질환다빈치로봇수술비(1회한)(갱신형)')


## We see that 47 chunks in total are created and well splitted.

##Indexing (categorization)

### Our chatbot's main objective is to provide appropriate message where the user queries are classified into several categories. In my opinion, to fully utilize the query categorization, I believe that each chunks (or blocks) should be categorized into those categories and stored as a field in metadata. Later by using this field, the chatbot will not only be dependent on the retrieved documents (which could include some chunks that are only related in the sense of cosine similarity - not in context similarity) but on the classified categories, which would work as a calibration for RAG system.  

### **But same as the metadata filling in the above code blocks, for the best performance, the work should be done by human - there still could be some hallucinations due to limited amount of contents in each chunks.

In [ ]:
import os
from typing import Dict, Literal

from openai import OpenAI
from pydantic import BaseModel

In [ ]:
LabelKey = Literal[
    "GREET",
    "GOODBYE",
    "FEEDBACK_POS",
    "FEEDBACK_NEG",
    "HUMAN_HANDOFF",
    "PRODUCT_OVERVIEW",
    "PLAN_OPTIONS",
    "COVERAGE_BENEFITS",
    "EXCLUSIONS_LIMITATIONS",
    "WAITING_PERIOD",
    "PREEXISTING_CONDITIONS",
    "ELIGIBILITY_UNDERWRITING",
    "PREMIUM_STABILITY_RENEWAL",
    "PRICING_GENERAL",
    "QUOTE_PERSONALIZED",
    "PAYMENT_BILLING",
    "COORDINATION_OTHER_INSURANCE",
    "APPLICATION_HOWTO",
    "CLAIM_FILING",
    "CLAIM_DOCUMENTS",
    "CLAIM_STATUS_TIMELINE",
    "POLICY_CHANGES_CANCEL_RENEW",
    "COMPLAINT_DISPUTE",
    "PRIVACY_PII",
    "MEDICAL_DIAGNOSIS_ADVICE",
    "OUT_OF_SCOPE",
]


LABELS_KO = {
    # Conversation (Static routing)
    "GREET": "인사",
    "GOODBYE": "대화 종료",
    "FEEDBACK_POS": "긍정 피드백/감사",
    "FEEDBACK_NEG": "부정 피드백/불만",
    "HUMAN_HANDOFF": "상담원 연결 요청",

    # Insurance (RAG routing)
    "PRODUCT_OVERVIEW": "보험 상품의 대략적인 개요에 대한 전반적인 설명",
    "PLAN_OPTIONS": "어떤 플랜/특약 옵션이 존재하는지 설명",
    "COVERAGE_BENEFITS": "보장 범위 (어떤 암까지 보장되는지) / 보장 혜택 / 지급 방식 / 암 발병 시 최대 보장 횟수",
    "EXCLUSIONS_LIMITATIONS": "보장하지 않는 손해(면책 사항) / 지급 제한 사항 / 중증도 기준",
    "WAITING_PERIOD": "대기 기간/면책 기간 (보험 가입 후 특정 질병이나 사고에 대해 보장이 시작되기 전까지의 기간)",
    "PREEXISTING_CONDITIONS": "기존에 있던 질병 / 병력에 따른 보험 조건",
    "ELIGIBILITY_UNDERWRITING": "보험 상품 가입 가능 여부 / 심사",
    "PREMIUM_STABILITY_RENEWAL": "보험료 고정/갱신",
    "PRICING_GENERAL": "전반적인 일반 보험료 / 가격",
    "QUOTE_PERSONALIZED": "개인의 현재 증상에 맞는 맞춤 견적",
    "PAYMENT_BILLING": "보험 상품 구매 시 납입 / 결제 / 청구",
    "COORDINATION_OTHER_INSURANCE": "타 보험과의 연계 / 중복",
    "APPLICATION_HOWTO": "보험 상품 가입 / 청약 방법",
    "CLAIM_FILING": "보험금 청구 방법",
    "CLAIM_DOCUMENTS": "청구 서류",
    "CLAIM_STATUS_TIMELINE": "청구 진행/처리 기간",
    "POLICY_CHANGES_CANCEL_RENEW": "계약 변경/해지/갱신",
    "COMPLAINT_DISPUTE": "민원/이의제기",

    # Safety filtering (Static routing)
    "PRIVACY_PII": "개인정보/민감정보 (고유번호나 주소 등)",
    "MEDICAL_DIAGNOSIS_ADVICE": "의료 상담/진단/치료 조언",
    "OUT_OF_SCOPE": "기타/범위 외 (보험과 관련이 거의 없는 질문)", #This should be routed to RAG
}

In [ ]:
class DocLabel(BaseModel):
    label: LabelKey
    reason: str


class LLMInsuranceDocLabeler:

    def __init__(self, model: str = "gpt-4o-mini-2024-07-18"):
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        self.model = model

        label_lines = "\n".join([f"- {k}: {v}" for k, v in LABELS_KO.items()])
        self.system_prompt = f"""너는 보험 약관/안내 문서의 '텍스트 조각(chunk)'을 아래의 라벨 중 하나로 분류해야 한다.
                              반드시 스키마에 맞춘 JSON만 출력하여라.

                              [라벨 목록]
                              {label_lines}

                              [규칙/예시]
                              - 문서 chunk가 인사/종료/피드백 등 대화행위가 아니라면 그런 라벨은 선택하지 마라.
                              - 의료 진단/치료 조언을 직접 제공하는 내용이면 MEDICAL_DIAGNOSIS_ADVICE.
                              - 개인정보(주민번호/계좌/연락처 등) 중심 내용이면 PRIVACY_PII.
                              - '청구 서류/구비서류'면 CLAIM_DOCUMENTS.
                              - '청구 방법/절차/접수'면 CLAIM_FILING.
                              - '처리 기간/지급까지 소요/진행 상태'면 CLAIM_STATUS_TIMELINE.
                              - '면책/제한/불지급'이면 EXCLUSIONS_LIMITATIONS.
                              - '대기기간/면책기간(시간 조건)'이면 WAITING_PERIOD.
                              - '보험료 고정/갱신/인상 구조'면 PREMIUM_STABILITY_RENEWAL.
                              - '보장/급부/지급 요건'이면 COVERAGE_BENEFITS.
                              - 애매하면 OUT_OF_SCOPE 대신, 가장 가까운 보험 라벨을 고르되 근거를 1문장으로 적어라.
                              """

    def label_chunk(self, page_content, block_category, block_title):
        user_prompt = f"""[metadata]
                        block_category: {block_category}
                        block_title: {block_title}

                        [chunk content]
                        {page_content}
                        """

        res = self.client.responses.parse(
            model=self.model,
            input=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            text_format=DocLabel,
            temperature=0,
            store=False,
        )
        parsed: DocLabel = res.output_parsed
        return {
            "insurance_label": parsed.label,
            "insurance_label_note": parsed.reason,
        }


### Now in the below, after inserting block_category and block_title field in metadata, I will also add a single summary line ("이 문서는 'block_category'로 분류되며, 다루는 내용/질문은 'block_title' 입니다") to the top of page content of each chunks to provide broader information about the chunk to the LLM that will later generate answers according to user's queries.

In [ ]:
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = api_key

labeler = LLMInsuranceDocLabeler(
    model="gpt-4o-mini-2024-07-18",
)

final_docs = []
for doc in split_docs:
    bm = doc.page_content
    bc = doc.metadata.get("block_category", "")
    bt = doc.metadata.get("block_title", "")

    out = labeler.label_chunk(bm, block_category=bc, block_title=bt)
    doc.metadata["insurance_label"] = out["insurance_label"]

    doc.page_content = (
        f"### 이 문서는 '{bc}'로 분류되며, 다루는 내용/질문은 '{bt}' 입니다\n"
        + re.sub(r"(?<!\.)\n", " ", doc.page_content)
    )
    final_docs.append(doc)

pprint(final_docs[0].metadata)
print(final_docs[0].page_content[:300])


{'author': 'U+ U+ U+ U+',
 'block_category': '<Q&A>',
 'block_title': '',
 'creationDate': "D:20251021141620+09'00'",
 'creationdate': '2025-10-21T14:16:20+09:00',
 'creator': 'Hwp 2020 11.0.0.8362',
 'end_page': 1,
 'file_path': '/content/drive/MyDrive/Hanhwa_files/hanhwa_100cancer_product_info.pdf',
 'format': 'PDF 1.4',
 'insurance_label': 'OUT_OF_SCOPE',
 'keywords': '',
 'modDate': "D:20251022082709+09'00'",
 'moddate': '2025-10-22T08:27:09+09:00',
 'page': 0,
 'producer': 'Hancom PDF 1.3.0.546',
 'source': '/content/drive/MyDrive/Hanhwa_files/hanhwa_100cancer_product_info.pdf',
 'start_page': 1,
 'subject': '',
 'title': '',
 'total_pages': 35,
 'trapped': ''}
### 이 문서는 '<Q&A>'로 분류되며, 다루는 내용/질문은 '' 입니다
##


### Now we see that at the metadata, 'insurance_label' field has been added with the corresponding category along with the summary line at the top of the page content.

### I will use Chroma db as the vector store for storing each chunk in vectorized form.

In [ ]:
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=final_docs[1:],
                                    embedding=embeddings_model,
                                    collection_name="hanhwa_insurance_case_qa",
                                    persist_directory="./chroma_db")

In [ ]:
# chroma_docs = vectorstore.similarity_search("연령별 가입 제한이 있나요?", k=5)
chroma_docs = vectorstore.similarity_search("10년납 해약환급금이 얼마야?", k=5)
for doc in chroma_docs:
    print(doc.page_content[:200], "\n")

### 이 문서는 '바. 해약환급금에 관한 사항'로 분류되며, 다루는 내용/질문은 '바-2. 해약환급금' 입니다
- ㅇ 가입기준 : 1종(납입면제형,기본형) 남자 40세, 상해1급, 월납 50,000원, 20년납 100세만기 ㅇ 보통약관 : 암(4대유사암제외)진단비 1,000만원 ㅇ 특별약관 : 상해사망 2,000만원, 4대유사암진단비 세부보장별 200만원 

### 이 문서는 '바. 해약환급금에 관한 사항'로 분류되며, 다루는 내용/질문은 '바-2. 해약환급금' 입니다
- ㅇ 가입기준 : 1종(납입면제형,기본형) 남자 40세, 상해1급, 월납 50,000원, 20년납 100세만기 ㅇ 보통약관 : 암(4대유사암제외)진단비 1,000만원 ㅇ 특별약관 : 상해사망 2,000만원, 4대유사암진단비 세부보장별 200만원 

### 이 문서는 '바. 해약환급금에 관한 사항'로 분류되며, 다루는 내용/질문은 '바-1. 해약환급금 산출기준' 입니다
- - 1종(납입면제형, 기본형)의 경우 회사는 금융감독원장이 인가한 산출기준에 따라 계산한 이 보험의 계약자적립액 에서 해약공제액을 공제한 금액을 해약환급금으로 지급하여 드립니다.   - - 2종(납입면제형, 납입후50%해약환급금지급형 

### 이 문서는 '바. 해약환급금에 관한 사항'로 분류되며, 다루는 내용/질문은 '바-1. 해약환급금 산출기준' 입니다
- - 1종(납입면제형, 기본형)의 경우 회사는 금융감독원장이 인가한 산출기준에 따라 계산한 이 보험의 계약자적립액 에서 해약공제액을 공제한 금액을 해약환급금으로 지급하여 드립니다.   - - 2종(납입면제형, 납입후50%해약환급금지급형 

### 이 문서는 '<Q&A>'로 분류되며, 다루는 내용/질문은 'Q. 해약환급금 일부지급형에 대해 설명해주세요.' 입니다
- 【2종(납입면제형, 납입후50%해약환급금지급형)   - ☞ 가. 납입후50%해약환급금지급형은 보험료 납입기간 중 계약이 해지될 경우 해약환급금을 지급하지 않으며, 보험료 납입이 완료되고 납입기간이 종료된 이후 

### I have tested how the embedding space actually works, and it shows that it well retrieves relative chunks that are closely related to the sample query.

## Classifier + RAG integration

### Now that all the documents and vector database has been prepared, we then finally create the complete pipeline:
1. Chatbot takes the user query (as a question).
2. The chatbot will classify the query to a certain category via LLM / regex filtering.
3. If the query belongs to a category that does not require LLM for generating answering, then the system will route to function which  automatically produces default sentence for computational efficiency.
4. Otherwise, the system will retrieve k chunks of documents from Croma db vector store based on vector similarity (cosine similarity) and categories (the query and retrieved document chunk should have the same category).
5. Based on the retrieved documents, the LLM will provide the appropriate answer to the user.

In [ ]:
LABEL_TO_HINT = {
    "CLAIM_FILING": "청구 절차를 단계별(1-2-3)로 정리하고 접수 채널/주의사항을 포함해라.",
    "CLAIM_DOCUMENTS": "필요 서류 목록 + 대체서류 + 자주 누락되는 항목을 포함해라.",
    "CLAIM_STATUS_TIMELINE": "처리기간(통상 범위) + 지연 사유 + 진행 확인 방법을 포함해라.",
    "EXCLUSIONS_LIMITATIONS": "면책/제한을 항목별로 정리하고 예외/판단기준을 명확히 해라.",
    "PREMIUM_STABILITY_RENEWAL": "갱신형/비갱신형 정의 후 보험료 인상 요인과 확인 포인트를 제시해라.",
}

DONT_RAG = {
    "GREET",
    "GOODBYE",
    "FEEDBACK_POS",
    "FEEDBACK_NEG",
    "HUMAN_HANDOFF",
    "PRIVACY_PII",
    "MEDICAL_DIAGNOSIS_ADVICE",
}

In [ ]:
import re
from dataclasses import dataclass
from typing import Any, Dict, Optional

from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from classifier import LLMQueryClassifier # This is from python file which classifies query into given categories (Task1, 2)
import pdb

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

class LangchainRAGbot:

    def __init__(self, vectorstore, model = "gpt-4o-mini-2024-07-18", k = 10):
        self.vectorstore = vectorstore
        self.k = k
        self.llm = ChatOpenAI(model=model, temperature=0)
        self.prompt = ChatPromptTemplate.from_template(
            """너는 보험 약관/상품 문서 기반의 상담 챗봇이다.

            아래 '근거 문서'에 포함된 내용만 사용해서 답변하라.
            근거가 부족하면, 어떤 정보가 더 필요한지 1~2개의 짧은 추가 질문을 하라.
            특정 라벨에 대하여는 답변 양식을 아래 '힌트'를 참조하여 답변하라.

            [힌트]
            {system_hint}

            [근거 문서]
            {context}

            [질문]
            {question}
            """
            )

    def answer(self, user_query, *,
        retrieval_query, metadata_filter, system_hint):

        q = retrieval_query or user_query

        docs = []
        try:
            search_kwargs = {"k": self.k}
            if metadata_filter:
                search_kwargs["filter"] = metadata_filter
            retriever = self.vectorstore.as_retriever(search_kwargs=search_kwargs)
            docs = retriever.invoke(q)
            # pdb.set_trace()
        except Exception: # Just for filter error
            docs = []

        if (not docs) and metadata_filter:
            retriever = self.vectorstore.as_retriever(search_kwargs={"k": self.k})
            docs = retriever.invoke(q)

        context = format_docs(docs) if docs else ""
        # print("문맥: ", context)
        hint = system_hint or ""

        chain = self.prompt | self.llm | StrOutputParser()
        # pdb.set_trace()
        return chain.invoke({"system_hint": hint, "context": context, "question": user_query})



In [ ]:
@dataclass
class OutFormat:
    label: str
    reason: str
    route: str
    answer: str
    retrieval_query: Optional[str] = None
    metadata_filter: Optional[Dict[str, Any]] = None

## Orchestration: Classifier → RAG

In [ ]:
class InsuranceAssistantOrchestrator:
    def __init__(self, classifier, rag):
        self.classifier = classifier
        self.rag = rag

    def static_reply(self, label):
        if label == "GREET":
            return "안녕하세요, 보험 관련해서 어떤 점이 궁금하신가요?"
        elif label == "GOODBYE":
            return "이용해 주셔서 감사합니다. 좋은 하루 보내세요."
        elif label == "FEEDBACK_POS":
            return "감사합니다! 추가로 궁금한 점이 있으면 편하게 말씀해 주세요."
        elif label == "FEEDBACK_NEG":
            return "불편을 드려 죄송합니다. 어떤 부분이 문제였는지 알려주시면 더 정확히 안내드릴게요."
        elif label == "HUMAN_HANDOFF":
            return "상담원 연결을 도와드릴게요. 잠시만 기다려 주세요."
        elif label == "PRIVACY_PII":
            return "개인정보(주민번호/계좌/연락처 등)는 입력하지 마세요. 개인정보 없이 질문을 다시 작성해 주세요."
        elif label == "MEDICAL_DIAGNOSIS_ADVICE":
            return (
                "의학적 진단/치료 조언은 제공할 수 없습니다. "
                "다만 보험 관점의 보장/청구/서류 요건 안내는 가능합니다. "
            )
        else:
          return "분류 없음. 무엇을 도와드릴까요?"

    def build_retrieval_query(self, user_query, label):
        return f"[{LABELS_KO.get(label, label)}] {user_query}"

    def build_metadata_filter(self, label):
        return {"insurance_label": label}

    def build_system_hint(self, label, reason):
        base = (
            f"분류 라벨: {label}({LABELS_KO.get(label, label)}). "
            f"분류 근거: {reason}. "
            f"답변은 이 라벨 관점에서 구체적으로 작성하라. "
        )
        extra = LABEL_TO_HINT.get(label, "")

        return base + extra

    def handle(self, user_query):
        c = self.classifier.classify(user_query)
        label = c.get("label", "OUT_OF_SCOPE") or "OUT_OF_SCOPE"
        reason = c.get("reason_ko", "")

        # Non-RAG routes
        if label in DONT_RAG:
            return OutFormat(
                label=label,
                reason=reason,
                route="STATIC",
                answer=self.static_reply(label),
            )

        # RAG route
        # pdb.set_trace()
        else:
          retrieval_query = self.build_retrieval_query(user_query, label)
          metadata_filter = self.build_metadata_filter(label)
          system_hint = self.build_system_hint(label, reason)

          answer = self.rag.answer(
              user_query,
              retrieval_query=retrieval_query,
              metadata_filter=metadata_filter,
              system_hint=system_hint,
          )

          return OutFormat(
              label=label,
              reason=reason,
              route="RAG",
              answer=answer,
              retrieval_query=retrieval_query,
              metadata_filter=metadata_filter,
          )


## Chatbot Demo

In [ ]:
classifier = LLMQueryClassifier(model="gpt-4o-mini-2024-07-18")
rag_bot = LangchainRAGbot(vectorstore=vectorstore, model="gpt-4o-mini-2024-07-18", k=10)
assistant = InsuranceAssistantOrchestrator(classifier=classifier, rag=rag_bot)

demo_queries = [
    "안녕하세요!",
    "상담원 연결해 주세요.",
    "10년납 해약환급금이 얼마야?",
    "상품 종류(종)는 어떻게 나뉘나요?",
    "제 이메일 주소가 hanhwa@gmail.com 인데 여기로 약정 보내주세요",
    "갱신 안내는 언제, 어떤 방식으로 오나요?",
    "아 오늘 점심은 뭘까요",
    "종료할게",
]

for q in demo_queries:
    out = assistant.handle(q)
    print("=" * 80)
    print("Q:", q)
    print("label:", out.label, "| reason:", out.reason)
    print("route:", out.route, "| filter:", out.metadata_filter)
    print("A:", out.answer)


Q: 안녕하세요!
label: GREET | reason: 로컬 규칙에 의해 분류되었습니다.
route: STATIC | filter: None
A: 안녕하세요, 보험 관련해서 어떤 점이 궁금하신가요?
Q: 상담원 연결해 주세요.
label: HUMAN_HANDOFF | reason: 로컬 규칙에 의해 분류되었습니다.
route: STATIC | filter: None
A: 상담원 연결을 도와드릴게요. 잠시만 기다려 주세요.
Q: 10년납 해약환급금이 얼마야?
label: POLICY_CHANGES_CANCEL_RENEW | reason: 해약환급금에 대한 질문이므로 계약 변경/해지와 관련된 내용으로 분류됨.
route: RAG | filter: {'insurance_label': 'POLICY_CHANGES_CANCEL_RENEW'}
A: 10년납의 해약환급금은 다음과 같습니다:

- 1년: 11,038원 (환급률 1.8%)
- 3년: 1,022,338원 (환급률 56.7%)
- 5년: 2,068,715원 (환급률 68.9%)
- 10년: 4,529,499원 (환급률 75.4%)

해약환급금은 계약 내용의 변경이나 보험료 실제 납입일자 등에 따라 달라질 수 있습니다. 추가로 궁금한 점이 있으신가요?
Q: 상품 종류(종)는 어떻게 나뉘나요?
label: PRODUCT_OVERVIEW | reason: 상품 종류에 대한 개요를 묻고 있음.
route: RAG | filter: {'insurance_label': 'PRODUCT_OVERVIEW'}
A: 이 상품은 무배당 상품으로 분류됩니다. 무배당 상품은 배당을 하지 않지만, 배당상품에 비해 보험료가 상대적으로 저렴하다는 특징이 있습니다. 추가로 궁금한 점이 있으신가요? 예를 들어, 특정 상품의 혜택이나 조건에 대해 알고 싶으신가요?
Q: 제 이메일 주소가 hanhwa@gmail.com 인데 여기로 약정 보내주세요
label: PRIVACY_PII | reason: 로컬 규칙에 의해 분류되었습니다.
route: 

### We could see that the chatbot is able to provide appropriate responses. It well utilizes the informations retrieved from the document that we provided and also filters out queries that contain contents not relevent to the insurance plans/products. However, we still could observe that the chatbot could not provide appropriate answer (ex: 상품 종류(종)는 어떻게 나뉘나요? --> The expected answer was 2(종) but the chatbot answers in an awkward way).

---
# **LangGraph Agentic Extension: Adaptive / Corrective Retrieval with a Grader Node**

### The pipeline above is a one-shot RAG system — it classifies/retrieves top-10 chunks, and generates without verifying whether the retrieved chunks are actually relevant to the specific question. The following cells extend the system into an agentic loop using LangGraph, implementing the CRAG-style (Corrective RAG) pattern described in the case study.

### The full graph has 6 nodes connected by a bounded retry loop:
1. **`classify_query`** — classifies the user query into an insurance label and sets the route

2. **`static_response`** — routes to return a fixed-template reply for non-RAG labels

3. **`retrieve`** — fetches top-k chunks from the vector store using the current retrieval query and metadata filter

4. **`grade_documents`** — scores each chunk with a cross-encoder reranker; if average relevance is below threshold, loops back

5. **`rewrite_query`** — rewrites the query (HyDE/synonyms on retry 1; drops filter + BM25 on retry 2) and increments `retry_count`

6. **`generate`** — produces the final answer from the graded chunks using the LLM


## **Phase 1 : Agent state setting & Edge routing function**

For efficient state transfer, here are the fields that will be included in the agent state:

- `user_query`       : Original user input.
- `retrieval_query`  : Query sent to the retriever.
- `label`            : Insurance category label from the classifier.
- `metadata_filter`  : Chroma filter dict --> ex: {"insurance_label":"COVERAGE_BENEFITS"}.
- `documents`        : Raw top-k docs returned by the retriever.
- `graded_documents` : Documents that passed the cross-encoder relevance threshold.
- `avg_relevance`    : Mean cross-encoder score across all retrieved docs (0.0–1.0).
- `retry_count`      : Number of rewrite/retry cycles completed so far (2).
- `route`            : "STATIC" for non-RAG labels, or "RAG".
- `answer`           : Final answer string.

In [ ]:
!pip install -U langgraph==1.1.8

In [100]:
from typing import TypedDict, List, Dict, Any
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END

In [101]:
class AgentState(TypedDict):
    user_query:        str
    retrieval_query:   str
    label:             str
    metadata_filter:   Dict[str, Any]
    documents:         List[Document]
    graded_documents:  List[Document]
    avg_relevance:     float
    retry_count:       int
    route:             str
    answer:            str

In [102]:
# Tunable constants
# These will be calibrated empirically during Phase 2 (Grader node)

RELEVANCE_THRESHOLD = 0.6
MAX_RETRIES         = 2

### Also, we add 2 conditional edge routing functions so that

- The STATIC & RAG based response generation could be routed in the agentic system
- The agent would be able to either generate the answer or loops back to rewrite the query based on the score & retrial threshold that we have set  

In [104]:
def route_decision(state: AgentState) -> str:
    """
    Branch on the route field.
    - STATIC --> static_response / - RAG --> retrieve
    """
    if state["route"] == "STATIC":
        return "static_response"
    return "retrieve"


def grade_decision(state: AgentState) -> str:
    """
    Decide whether to generate or loop back for a retry.
    - avg_relevance >= RELEVANCE_THRESHOLD  --> generate
    - retry_count   >= MAX_RETRIES          --> generate
    - otherwise                             --> rewrite_query
    """
    passed    = state["avg_relevance"] >= RELEVANCE_THRESHOLD
    exhausted = state["retry_count"]   >= MAX_RETRIES
    if passed or exhausted:
        return "generate"
    return "rewrite_query"

## **Phase 2 — Grader Node: Cross-Encoder Relevance Scoring**

### The one-shot pipeline retrieved top-10 chunks by cosine similarity and sent all of them to the LLM. The core problem: cosine similarity is the same metric used for retrieval, so it cannot independently verify whether a chunk actually answers the question. A chunk can be geometrically close to the query embedding but semantically off-topic.

In [107]:
!pip install -q sentence-transformers

In [108]:
from sentence_transformers import CrossEncoder
import numpy as np


def _sigmoid(x):
    """Maps raw cross-encoder logits to [0, 1]."""
    return 1.0 / (1.0 + np.exp(-np.asarray(x, dtype=float)))


# CrossEncoder reads query and passage jointly through shared attention
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)
print("Cross-encoder loaded:", type(reranker).__name__)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Cross-encoder loaded: CrossEncoder


In [110]:
# Docs that score below this on the cross-encoder are excluded from
# graded_documents, so they never reach the LLM.
PER_CHUNK_THRESHOLD = 0.45


def node_grade_documents(state: AgentState) -> dict:
    """
    Scores each doc in state["documents"] against the current retrieval_query
    using the bge-reranker-v2-m3 cross-encoder (reads query + passage jointly).
    """

    docs  = state["documents"]
    query = state["retrieval_query"] or state["user_query"]

    if not docs:
        return {"graded_documents": [], "avg_relevance": 0.0}

    # CrossEncoder.predict() takes a list of (query, passage) tuples
    pairs = [(query, doc.page_content) for doc in docs]

    # sigmoid normalises to [0, 1]
    raw_scores = _sigmoid(reranker.predict(pairs)).tolist()

    avg_relevance = sum(raw_scores) / len(raw_scores)

    graded_docs = [
        doc for doc, score in zip(docs, raw_scores)
        if score >= PER_CHUNK_THRESHOLD
    ]

    return {
        "graded_documents": graded_docs,
        "avg_relevance":    avg_relevance,
    }

## **Phase 3 — Query Rewriting Node**

### When average relevance score is lower than setted threshold after grading, the graph loops back to "rewrite the query" rather than proceeding to generate. This node applies one of two correction strategies based on `retry_count` at the time of the call.
<br>

### **Retry 1 (`retry_count` 0 → 1): HyDE + Jargon/Synonym Expansion**

GPT-4o-mini rewrites the query by combining two techniques in a single call:

- **HyDE (Hypothetical Document Embedding)**: generates 1–2 sentences of a hypothetical answer passage --> text that would appear in the insurance document if the answer were there. Embedding this richer passage surfaces chunks that the short query missed.

- **Jargon/synonym expansion**: expands insurance-specific terms with colloquial alternatives (ex: "납입면제" → "보험료 납입 면제 조건"), broadening the match surface in the embedding space.

The `metadata_filter` stays still at retry 1 as we are still searching within the correct label's chunk pool.
<br>

### **Retry 2 (`retry_count` 1 → 2): Filter Relaxation**

If retry 1 still fails the grader, `metadata_filter` is dropped to blank, opening the search to all 47 chunks across all labels. The HyDE-rewritten query from retry 1 is carried forward. We also activate BM25 hybrid search on this path to handle term-heavy queries that dense retrieval struggles with.

In [113]:
import os
from openai import OpenAI

In [114]:
def node_rewrite_query(state: AgentState) -> dict:

    current_retry = state["retry_count"]

    if current_retry == 0:
        # Retry 1: HyDE + jargon expansion
        rewrite_prompt = f"""너는 한국 보험 상품 문서 검색 전문가다.
사용자의 원래 질문을 보고, 문서 검색에 더 효과적인 새로운 검색 쿼리를 만들어라.

[원래 질문]
{state['user_query']}

[분류된 라벨]
{state['label']}

다음 두 가지를 결합하여 하나의 검색 쿼리를 만들어라:
1. 질문에 나오는 보험 전문 용어를 동의어 및 구어체 표현으로 확장
   (예: "해약환급금" → "해지 시 돌려받는 금액, 해약환급금, 환급률")
2. 이 질문에 대한 보험 약관 문서에 실제로 있을 법한 내용을 1~2문장으로 가상의 답변으로 작성 (HyDE)

검색 쿼리만 출력하고 다른 설명은 하지 마라."""

        client   = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        response = client.chat.completions.create(
            model       = "gpt-4o-mini-2024-07-18",
            messages    = [{"role": "user", "content": rewrite_prompt}],
            temperature = 0.3,
            max_tokens  = 200,
        )
        new_query = response.choices[0].message.content.strip()

        return {
            "retrieval_query": new_query,
            "metadata_filter": state["metadata_filter"],   # label-strict, unchanged
            "retry_count":     current_retry + 1,
        }

    else:
        # Retry 2: drop filter → full-corpus search
        # Carry forward the HyDE-rewritten query from retry 1.
        return {
            "retrieval_query": state["retrieval_query"],
            "metadata_filter": {},                          # full-corpus, no label restriction
            "retry_count":     current_retry + 1,
        }

## Phase 4 — Hybrid Search Infrastructure (BM25 + RRF)

### The retry-2 path drops `metadata_filter` to blank and searches the full chunk corpus. Dense vector retrieval alone can still miss term-heavy queries like exact Korean terms that is finely shown in the document but whose embedding drifts from the query embedding. We will implement BM25 to fill in that gap.

1. Build a **BM25Okapi** index over all chunks corpus using whitespace tokenisation
2. Implement **Reciprocal Rank Fusion (RRF)** to merge the BM25 and dense ranked lists into a single ranked result — each document's score is `Σ 1 / (rank + RRF_K)` rewarding documents that rank highly in multiple sources
3. Implement **hybrid_search** which will be called only on the 2nd retry path when `metadata_filter` is blank

In [117]:
!pip install -q rank-bm25

In [118]:
from rank_bm25 import BM25Okapi


def tokenize_ko(text: str) -> list:
    """
    Minimal Korean tokenizer: lowercase + whitespace split.
    """
    return [t for t in text.lower().split() if t]


# Build BM25Okapi index over all chunks in final_docs.
_bm25_corpus = [tokenize_ko(doc.page_content) for doc in final_docs[1:]]
bm25_index   = BM25Okapi(_bm25_corpus)

print(f"BM25 index built: {len(_bm25_corpus)} chunks | "
      f"vocab ≈ {len(bm25_index.idf)} unique tokens.")

BM25 index built: 48 chunks | vocab ≈ 2235 unique tokens.


In [119]:
TOP_K = 10   # chunks to retrieve per search pass
RRF_K = 60   # RRF smoothing constant


def reciprocal_rank_fusion(ranked_lists: list, top_k: int = TOP_K) -> list:
    """
    Merge multiple ranked document lists with Reciprocal Rank Fusion.

    Documents absent from a list receive no contribution from it.
    The 120-char page_content prefix is used as a stable document key.
    """
    scores  = {}
    doc_map = {}

    for ranked_list in ranked_lists:
        for rank, doc in enumerate(ranked_list):
            key = doc.page_content[:120]
            if key not in scores:
                scores[key]  = 0.0
                doc_map[key] = doc
            scores[key] += 1.0 / (rank + 1 + RRF_K)

    merged = sorted(scores.keys(), key=lambda k: scores[k], reverse=True)
    return [doc_map[k] for k in merged[:top_k]]


def hybrid_search(query: str, k: int = TOP_K) -> list:
    """
    Blend BM25 (lexical) and Chroma dense (semantic) retrieval via RRF.
    Activated only on the 2nd retry path where metadata_filter is blank.
    """
    # BM25 retrieval
    bm25_scores  = bm25_index.get_scores(tokenize_ko(query))
    top_bm25_idx = sorted(range(len(bm25_scores)),
                          key=lambda i: bm25_scores[i], reverse=True)[:k]
    bm25_docs    = [final_docs[i] for i in top_bm25_idx]

    # Dense retrieval (no filter, full corpus)
    dense_docs = vectorstore.similarity_search(query, k=k)

    # Merge via RRF
    return reciprocal_rank_fusion([bm25_docs, dense_docs], top_k=k)

In [120]:
def node_retrieve(state: AgentState) -> dict:

    query  = state["retrieval_query"] or state["user_query"]
    filter = state["metadata_filter"]

    if filter:
        # Dense-only with metadata label filter
        try:
            docs = vectorstore.similarity_search(query, k=TOP_K, filter=filter)
        except Exception:
            docs = []
        # filter too restrictive: full-corpus dense search
        if not docs:
            docs = vectorstore.similarity_search(query, k=TOP_K)
    else:
        # Hybrid: BM25 + dense merged via RRF
        docs = hybrid_search(query, k=TOP_K)

    return {"documents": docs}

## Phase 5 — Graph Assembly, Integration & End-to-End Testing

### All node implementations from Phases 2–4 are in place. This final phase uses the existing components from the original LangChain pipeline to fill in the missing parts for full pipeline, compiles the final graph with all 6 nodes real, and runs end-to-end tests across 20 representative queries.
</br>

1. **`node_classify_query`** — delegates to the existing `LLMQueryClassifier`.

2. **`node_static_response`** — delegates to `InsuranceAssistantOrchestrator`.

3. **`node_generate`** — reuses the same GPT-4o-mini prompt from `LangchainRAGbot` but now receives `graded_documents` (cross-encoder-filtered) instead of the raw top-10.

4. **Final graph compilation** — all 6 nodes are compiled into single graph

5. **20-query E2E test** — the original 8 LangChain demo queries plus 12 added queries covering all remaining label categories; each result shows `label`, `route`, `avg_relevance`, `retry_count`, and graded doc count alongside the answer.

In [123]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [124]:
def node_classify_query(state: AgentState) -> dict:
    """
    Delegates to LLMQueryClassifier.classify():
      1. Heuristic regex — no LLM call, reason_ko is set to "로컬 규칙에 의해 분류되었습니다."
      2. GPT-4o-mini structured output via Pydantic schema — for all other queries,
         reason_ko contains the classifier's Korean explanation.
    """
    c      = classifier.classify(state["user_query"])
    label  = c.get("label", "OUT_OF_SCOPE") or "OUT_OF_SCOPE"
    reason = c.get("reason_ko", "")

    route           = "STATIC" if label in DONT_RAG else "RAG"
    retrieval_query = f"[{LABELS_KO.get(label, label)}] {state['user_query']}"
    metadata_filter = {"insurance_label": label} if route == "RAG" else {}

    return {
        "label":           label,
        "reason":          reason,
        "route":           route,
        "retrieval_query": retrieval_query,
        "metadata_filter": metadata_filter,
    }

In [125]:
def node_static_response(state: AgentState) -> dict:
    """
    Delegates to InsuranceAssistantOrchestrator.static_reply(), which returns
    fixed-template Korean replies for non-RAG labels
    """
    return {"answer": assistant.static_reply(state["label"])}

In [126]:
# Reuse the exact same system prompt as LangchainRAGbot — only the context
# source changes: graded_documents instead of raw top-10.
_generate_prompt = ChatPromptTemplate.from_template(
    """너는 보험 약관/상품 문서 기반의 상담 챗봇이다.

아래 '근거 문서'에 포함된 내용만 사용해서 답변하라.
근거가 부족하면, 어떤 정보가 더 필요한지 1~2개의 짧은 추가 질문을 하라.
특정 라벨에 대하여는 답변 양식을 아래 '힌트'를 참조하여 답변하라.

[힌트]
{system_hint}

[근거 문서]
{context}

[질문]
{question}"""
)
_llm = ChatOpenAI(model="gpt-4o-mini-2024-07-18", temperature=0)


def node_generate(state: AgentState) -> dict:
    """
    Key difference from LangchainRAGbot.answer(): retrieval is already done
    and graded upstream (this node is pure generation only).
    """
    docs    = state["graded_documents"] if state["graded_documents"] else state["documents"]
    label   = state["label"]
    reason  = state.get("reason", "")
    context = format_docs(docs) if docs else ""

    # Mirror build_system_hint(): label + classifier reason + label-specific hint
    system_hint = (
        f"분류 라벨: {label}({LABELS_KO.get(label, label)}). "
        + (f"분류 근거: {reason}. " if reason else "")
        + f"답변은 이 라벨 관점에서 구체적으로 작성하라. "
        + LABEL_TO_HINT.get(label, "")
    )

    chain  = _generate_prompt | _llm | StrOutputParser()
    answer = chain.invoke({
        "system_hint": system_hint,
        "context":     context,
        "question":    state["user_query"],
    })
    return {"answer": answer}

In [127]:
# Final graph: all 6 nodes are real implementations

builder = StateGraph(AgentState)

builder.add_node("classify_query",  node_classify_query)   # Phase 5
builder.add_node("static_response", node_static_response)  # Phase 5
builder.add_node("retrieve",        node_retrieve)          # Phase 4
builder.add_node("grade_documents", node_grade_documents)   # Phase 2
builder.add_node("rewrite_query",   node_rewrite_query)     # Phase 3
builder.add_node("generate",        node_generate)          # Phase 5

builder.add_edge(START,             "classify_query")
builder.add_edge("retrieve",        "grade_documents")
builder.add_edge("rewrite_query",   "retrieve")
builder.add_edge("static_response", END)
builder.add_edge("generate",        END)

builder.add_conditional_edges(
    "classify_query",
    route_decision,
    {"static_response": "static_response", "retrieve": "retrieve"},
)
builder.add_conditional_edges(
    "grade_documents",
    grade_decision,
    {"generate": "generate", "rewrite_query": "rewrite_query"},
)

graph = builder.compile()
print("Final graph compiled — all 6 nodes are real implementations.")
print("Node summary:")
print("  classify_query  → LLMQueryClassifier          (Phase 5)")
print("  static_response → InsuranceAssistantOrchestrator.static_reply  (Phase 5)")
print("  retrieve        → Chroma dense / BM25+RRF hybrid               (Phase 4)")
print("  grade_documents → bge-reranker-v2-m3 cross-encoder             (Phase 2)")
print("  rewrite_query   → HyDE+jargon / filter relaxation              (Phase 3)")
print("  generate        → GPT-4o-mini on graded_documents              (Phase 5)")

Final graph compiled — all 6 nodes are real implementations.
Node summary:
  classify_query  → LLMQueryClassifier          (Phase 5)
  static_response → InsuranceAssistantOrchestrator.static_reply  (Phase 5)
  retrieve        → Chroma dense / BM25+RRF hybrid               (Phase 4)
  grade_documents → bge-reranker-v2-m3 cross-encoder             (Phase 2)
  rewrite_query   → HyDE+jargon / filter relaxation              (Phase 3)
  generate        → GPT-4o-mini on graded_documents              (Phase 5)


In [129]:
# ── End-to-End Test: 20 queries across all label categories ──────────────────
# Queries 1–8  : the original 8 LangChain demo queries (kept exactly)
# Queries 9–20 : 12 additional queries covering remaining label categories

e2e_queries = [
    # ── Original 8 LangChain demo queries ────────────────────────────────────
    "안녕하세요!",                                                        # GREET
    "상담원 연결해 주세요.",                                               # HUMAN_HANDOFF
    "10년납 해약환급금이 얼마야?",                                         # POLICY_CHANGES_CANCEL_RENEW
    "상품 종류(종)는 어떻게 나뉘나요?",                                    # PRODUCT_OVERVIEW
    "제 이메일 주소가 hanhwa@gmail.com 인데 여기로 약정 보내주세요",        # PRIVACY_PII
    "갱신 안내는 언제, 어떤 방식으로 오나요?",                             # PREMIUM_STABILITY_RENEWAL
    "아 오늘 점심은 뭘까요",                                               # OUT_OF_SCOPE
    "종료할게",                                                            # GOODBYE
    # ── 12 additional queries covering remaining label categories ─────────────
    "하이클래스 항암약물치료비 갱신형 연간 지급",
    "감사합니다, 많은 도움이 됐어요",                                       # FEEDBACK_POS
    "설명이 너무 어렵고 불친절한 것 같아요",                               # FEEDBACK_NEG
    "제가 암 진단을 받았는데 어떤 치료를 받아야 하나요?",                  # MEDICAL_DIAGNOSIS_ADVICE
    "어떤 특약 옵션들이 있나요?",                                          # PLAN_OPTIONS
    "암 진단 시 얼마를 받을 수 있나요?",                                   # COVERAGE_BENEFITS
    "보장하지 않는 암 종류가 있나요?",                                     # EXCLUSIONS_LIMITATIONS
    "가입 후 언제부터 보장이 시작되나요?",                                 # WAITING_PERIOD
    "기존 병력이 있어도 가입이 가능한가요?",                               # PREEXISTING_CONDITIONS
    "가입 가능한 나이 범위가 어떻게 되나요?",                              # ELIGIBILITY_UNDERWRITING
    "보험금은 어떻게 청구하나요?",                                         # CLAIM_FILING
    "보험금 청구에 필요한 서류가 뭔가요?",                                 # CLAIM_DOCUMENTS
    "보험금 처리 기간이 얼마나 걸리나요?",                                 # CLAIM_STATUS_TIMELINE
]

_blank: AgentState = {
    "user_query": "", "retrieval_query": "", "label": "", "reason": "",
    "metadata_filter": {}, "documents": [], "graded_documents": [],
    "avg_relevance": 0.0, "retry_count": 0, "route": "", "answer": "",
}

results = []
for idx, query in enumerate(e2e_queries, 1):
    result = graph.invoke({**_blank, "user_query": query})
    results.append(result)

    print("=" * 80)
    print(f"[{idx:02d}] Q : {query}")
    print(f"      label  : {result['label']:<30}  route : {result['route']}")

    reason = result.get("reason", "")
    if reason:
        print(f"      reason : {reason}")

    if result["route"] == "RAG":
        print(f"      avg_relevance : {result['avg_relevance']:.3f}"
              f"  |  retries : {result['retry_count']}"
              f"  |  graded_docs : {len(result['graded_documents'])}")

    print(f"      A :")
    print(result["answer"] if result["answer"] else "(no answer)")

n_static = sum(1 for r in results if r["route"] == "STATIC")
n_rag    = sum(1 for r in results if r["route"] == "RAG")
print("=" * 80)
print(f"E2E test complete — {len(e2e_queries)} queries | STATIC: {n_static} | RAG: {n_rag}")

[01] Q : 안녕하세요!
      label  : GREET                           route : STATIC
      A :
안녕하세요, 보험 관련해서 어떤 점이 궁금하신가요?
[02] Q : 상담원 연결해 주세요.
      label  : HUMAN_HANDOFF                   route : STATIC
      A :
상담원 연결을 도와드릴게요. 잠시만 기다려 주세요.
[03] Q : 10년납 해약환급금이 얼마야?
      label  : POLICY_CHANGES_CANCEL_RENEW     route : RAG
      avg_relevance : 0.555  |  retries : 2  |  graded_docs : 10
      A :
10년납의 해약환급금은 다음과 같습니다:

- 1종(납입면제형, 기본형) 해약환급금: 4,529,499원 (환급률 75.4%)
- 2종(납입후50%해약환급금지급형) 해약환급금: 4,674,649원 (환급률 77.9%)

해약환급금은 계약 내용의 변경, 보험료 실제 납입일자 등에 따라 달라질 수 있습니다. 추가로 궁금한 점이 있으신가요?
[04] Q : 상품 종류(종)는 어떻게 나뉘나요?
      label  : PRODUCT_OVERVIEW                route : RAG
      avg_relevance : 0.502  |  retries : 2  |  graded_docs : 10
      A :
이 상품은 두 가지 종류로 나뉩니다:

1. **1종(납입면제형, 기본형)**: 이 상품은 순수보장성 보험으로, 만기환급금이 지급되지 않습니다. 
2. **2종(납입면제형, 납입후50%해약환급금지급형)**: 이 상품 역시 순수보장성 보험으로, 중도인출이 불가능하며, 보험료 납입이 완료된 후 계약이 해지될 경우 해약환급금의 50%가 지급됩니다.

추가로 궁금한 점이 있으신가요? 예를 들어, 각 상품의 가입 조건이나 보장 내용에 대해 알고 싶으

### We now could clearly see that for the queries that the original RAG system could not provide answers properly, Our agentic pipeline could now generate correct responses.